In [ ]:
import os
import torch
import pandas as pd
import librosa
import numpy as np
import json
from torch.utils.data import Dataset
from transformers import (
    SpeechT5Processor, 
    SpeechT5ForTextToSpeech, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    SpeechT5FeatureExtractor,
    BertTokenizer
)
from dataclasses import dataclass
from typing import Any, Dict, List

# 1. PATHS & CONFIG

LOCAL_MODEL_PATH = r"C:/TTS-Speech_t5/speecht5-tam" 
DATASET_DIR = r"C:/TTS-Speech_t5/tamil_dataset"

WAV_DIR = r"C:/TTS-Speech_t5/tamil_dataset/wavs"
OUTPUT_DIR = r"C:/TTS-Speech_t5/tamil_tts_output"

device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. PREPROCESSOR CHECK

prepro_path = os.path.join(LOCAL_MODEL_PATH, "preprocessor_config.json")
if not os.path.exists(prepro_path):
    config = {
        "feature_extractor_type": "SpeechT5FeatureExtractor",
        "feature_size": 1, "num_mel_bins": 80, "hop_length": 256,
        "win_length": 1024, "sampling_rate": 16000, "normalize_speech": True,
        "do_normalize": True, "return_attention_mask": True
    }
    with open(prepro_path, "w") as f:
        json.dump(config, f)

# 3. LOAD MODEL & PROCESSOR

feature_extractor = SpeechT5FeatureExtractor.from_pretrained(LOCAL_MODEL_PATH)
tokenizer = BertTokenizer.from_pretrained(LOCAL_MODEL_PATH)
processor = SpeechT5Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
model = SpeechT5ForTextToSpeech.from_pretrained(LOCAL_MODEL_PATH).to(device)

# 4. CORRECTED TAMIL DATASET CLASS

class TamilTTSDataset(Dataset):
    def __init__(self, csv_file, processor, wav_dir):
        self.df = pd.read_csv(csv_file)
        self.processor = processor
        self.wav_dir = os.path.abspath(wav_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["sentence"]) if pd.notna(row["sentence"]) else ""
        # Build the absolute path to the audio file
        audio_filename = str(row["audio"]).strip()
        audio_path = os.path.join(self.wav_dir, audio_filename)
        
        # Load audio 
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Could not find audio at: {audio_path}")
            
        speech, _ = librosa.load(audio_path, sr=16000)
        
        # Process Tamil text and audio
        inputs = self.processor(
            text=text,
            audio_target=speech,
            sampling_rate=16000,
            return_attention_mask=False,
        )

        return {
            "input_ids": inputs["input_ids"],
            "labels": torch.tensor(inputs["labels"][0]),
            "speaker_embeddings": torch.zeros(512, dtype=torch.float32) 
        }

# Initialize Datasets with the wav_dir
train_ds = TamilTTSDataset(os.path.join(DATASET_DIR, "train.csv"), processor, WAV_DIR)
test_ds = TamilTTSDataset(os.path.join(DATASET_DIR, "validation.csv"), processor, WAV_DIR)

# 5. DATA COLLATOR

@dataclass
class TTSDataCollatorWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        label_features = [{"input_values": feature["labels"]} for feature in features]
        speaker_features = [feature["speaker_embeddings"] for feature in features]

        # Pad input text and audio labels
        batch = self.processor.pad(input_ids=input_ids, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, return_tensors="pt")
        
        labels = labels_batch["input_values"]
        # SpeechT5 requires even number of frames
        if labels.shape[1] % 2 != 0:
            labels = labels[:, :-1, :]

        batch["labels"] = labels
        # Stack speaker embeddings correctly
        batch["speaker_embeddings"] = torch.stack(speaker_features)
        
        return batch

collator = TTSDataCollatorWithPadding(processor=processor)

# 6. TRAINING CONFIGURATION

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000, 
    save_steps=500,
    eval_strategy="steps",
    fp16=True if torch.cuda.is_available() else False,
    logging_steps=10,
    remove_unused_columns=False,
    label_names=["labels"]
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
)

# 7. EXECUTION

print("Starting Tamil SpeechT5 Fine-Tuning...")
trainer.train()

# Save final model
final_path = os.path.join(OUTPUT_DIR, "final_tamil_speecht5")
trainer.save_model(final_path)
processor.save_pretrained(final_path)
print(f"✅ Success! Model saved to {final_path}")

Loading weights: 100%|██████████| 396/396 [00:00<00:00, 870.28it/s, Materializing param=speecht5.encoder.wrapped_encoder.layers.11.layer_norm.weight]                     


Starting Tamil SpeechT5 Fine-Tuning...


Step,Training Loss,Validation Loss
10,24.520015,3.267786
20,22.129921,2.827357
30,19.016324,2.262649
40,15.810225,1.901949
50,13.738794,1.319832
60,10.991946,0.999302
70,8.487559,0.857451
80,7.584684,0.725742
90,6.681931,0.631546
100,5.992383,0.614598


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.31s/it]

✅ Success! Model saved to C:/TTS-Speech_t5/tamil_tts_output\final_tamil_speecht5
